In [1]:
import os
import yaml

from typing import TypedDict, Annotated, List, Tuple, Literal, Union
from enum import Enum
from pydantic import BaseModel, Field
from dataclasses import dataclass
import operator

from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'code' / 'tool'))
from rag_tool import create_rag_tool
from text2sql_tool import text2sql_workflow

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
def load_prompt(yaml_file):
    with open(yaml_file, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

inmemory_saver = InMemorySaver()

### State

In [4]:
class MyAgentState(TypedDict):
    input: Annotated[str, "User's input"]
    
    intent_category: Annotated[str, "Intent category"]
    rewritten_query: Annotated[str, "Rewritten query"]

    chat_history: Annotated[List[BaseMessage], operator.add]

### 1. Intent Analyze

In [5]:
class Category(str, Enum):
    """Defines the query processing categories."""
    COMPLEX = "COMPLEX"
    SIMPLE = "SIMPLE"
    INAPPROPRIATE = "INAPPROPRIATE"

class Intent(BaseModel):
    """Schema containing the user query’s intent, processing category, and rewritten query."""
    category: Category = Field(
        description="Query processing category. Must be one of 'COMPLEX', 'SIMPLE', or 'INAPPROPRIATE'."
    )
    query_rewrite: str = Field(
        description="A rewritten version of the original query to make it easier for downstream Agents to process. If the category is INAPPROPRIATE, contains a rejection message."
    )

parser = PydanticOutputParser(pydantic_object=Intent)

In [6]:
with_history_prompt = load_prompt('../prompt/intent_analyze/with_history_20251127_01.yaml')
with_history_prompt = PromptTemplate(
    template=with_history_prompt['template'],
    input_variables=with_history_prompt['input_variables']
)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

with_history_prompt = with_history_prompt.partial(format=parser.get_format_instructions())
with_chain = with_history_prompt | llm

In [7]:
def intentAnalyze(state: MyAgentState, config: RunnableConfig):
    configurable = config.get("configurable", {})
    thread_id = configurable.get("thread_id")
    user_id = configurable.get("user_id")

    result = with_chain.invoke({'user_query': state['input'],
                                'chat_history': state['chat_history'],
                                'user_id': user_id})

    structed_output = parser.parse(result.content)
    return {'intent_category': structed_output.category.value,
            'rewritten_query': structed_output.query_rewrite}

In [8]:
def should_continue_intent(state: MyAgentState) -> Literal['COMPLEX', 'SIMPLE', 'INAPPROPRIATE']:
    return state.get('intent_category', 'SIMPLE')

### 2. Tool

In [9]:
rag_tool = create_rag_tool()
text2sql_app = text2sql_workflow(checkpointer=inmemory_saver)

In [10]:
@tool
def query_my_scores(query: str, runtime: ToolRuntime) -> str:
    """
    Converts the user's natural language query into SQL, retrieves the data from the database.

    Args:
        query: The user's natural language question.

    When to use:
    - When the user wants to view their exercise analysis data.
    - When querying for scores, trends, statistics, or other information stored in the database.
    - Examples: "What is the upper body score of my latest impression video?"

    Returns:
        The final answer or an error message.
    """
    config = runtime.config
    thread_id = config['configurable']['thread_id'] + '_text2sql'
    user_id = config['configurable']['user_id']

    sub_config = {
        "configurable": {
            "thread_id": thread_id,
            "user_id": user_id
        }
    }
    
    try:
        initial_state = {
            "origin_query": query,
            "rewritten_query": [],
            "errors": [],
            "sql": [],
            "selected_data": [],
            "final_answer": "",
            "cancelled": False,
            "error_num": 0,
            "direction": "",
            "only_data": True
        }
        
        result = text2sql_app.invoke(initial_state, config=sub_config)
        
        return result.get("final_answer", "No answer generated")
    
    except Exception as e:
        return f"Error executing query: {str(e)}"

### 3. Complex(to do list)


In [11]:
@dataclass
class AgentContext:
    thread_id: str
    user_id: str

In [23]:
tmp_systemprompt = """
    당신은 역도 코칭 에이전트입니다.
"""

In [30]:
execute_todo_agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[rag_tool, query_my_scores],
    context_schema=AgentContext,
    middleware=[
        TodoListMiddleware(
            system_prompt="""Always start by creating the next plan with write_todos."""
        )
    ]
)

In [31]:
sub_config = {
    "configurable": {
        "thread_id": '1',
        "user_id": '1'
    }
}

execute_todo_agent.invoke(
    {"messages": [("user", "나의 최근 인상 영상 중 가장 문제가 되는 신체부위를 알려줘. 그리고 신체부위 관련 옳바른 자세를 설명해줘.")]},
    config=sub_config
)

{'messages': [HumanMessage(content='나의 최근 인상 영상 중 가장 문제가 되는 신체부위를 알려줘. 그리고 신체부위 관련 옳바른 자세를 설명해줘.', additional_kwargs={}, response_metadata={}, id='2cd5e628-1c54-469b-8e91-bcfdea7f856c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 1092, 'total_tokens': 1153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D4tHjic9TozQmSoPIim7WRffRU9id', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--6e0929ef-8cde-4a96-8561-fbd3171ec8af-0', tool_calls=[{'name': 'query_my_scores', 'args': {'query': 'What is the problematic body part in my latest impression video?'}, 'id': 'call_0m8CE6uTYKgJAsJIc

In [32]:
for chunk in execute_todo_agent.stream(
    {"messages": [("user", "나의 최근 인상 영상 중 가장 문제가 되는 신체부위를 알려줘. 그리고 신체부위 관련 옳바른 자세를 설명해줘. 그리고 적절한 훈련 계획을 수립해줘.")]},
    config=sub_config,
    stream_mode="updates"
):
    for node_name, values in chunk.items():
        print(f"Node: {node_name}, Keys: {list(values.keys())}")  # 먼저 키 확인!
        
        if "planning_state" in values:  # planning_state 키 확인
            planning_state = values["planning_state"]
            todos = planning_state.get("todos", [])
            if todos:
                for i, todo in enumerate(todos):
                    status = todo.get('status', 'pending')
                    icon = "✅" if status == "completed" else "⏳" if status == "in_progress" else "💤"
                    print(f"  {icon} {i+1}. {todo['content']}")
                print("-" * 30)

Node: model, Keys: ['messages']
Node: tools, Keys: ['messages']
Node: model, Keys: ['messages']
Node: tools, Keys: ['messages']
Node: tools, Keys: ['messages']
Node: model, Keys: ['messages']
